In [36]:
%pwd

'C:\\Users\\yadav\\OneDrive\\Desktop\\GenAI Project\\End-to-End-Medical-Chatbot'

In [37]:
import os

os.chdir("../")

In [38]:
%cd "C:\Users\yadav\OneDrive\Desktop\GenAI Project\End-to-End-Medical-Chatbot"

C:\Users\yadav\OneDrive\Desktop\GenAI Project\End-to-End-Medical-Chatbot


In [39]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
# Extract Data from PDF files

def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob = "*.pdf",
        loader_cls = PyPDFLoader
    )

    documents = loader.load()
    return documents


In [41]:
extracted_docs = load_pdf_files("data")

In [42]:
extracted_docs[20:22]

[Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book.pdf', 'total_pages': 637, 'page': 20, 'page_label': '21'}, page_content='the late second or third trimester. Although D&X is high-\nly controversial, some physicians argue that it has advan-\ntages that make it a preferable procedure in some circum-\nstances. One perceived advantage is that the fetus is\nremoved largely intact, allowing for better evaluation and\nautopsy of the fetus in cases of known fetal anomalies.\nIntact removal of the fetus may also confer a lower risk\nof puncturing the uterus or damaging the cervix. Another\nperceived advantage is that D&X ends the pregnancy\nwithout requiring the woman to go through labor, which\nmay be less emotionally traumatic than other methods of\nlate-term abortion. In addition, D&X may offer a lower\ncost and shorter procedure time.\nPrecautions

In [43]:
len(extracted_docs)

637

In [44]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    '''
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and original page_content.
    '''

    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata = {"source": src}
            )
        )
    
    return minimal_docs

minimal_docs = filter_to_minimal_docs(extracted_docs)

In [45]:
minimal_docs[:6]

[Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content=''),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow,Manager, Imaging and Multimedia\nContent\nRobyn V . Young,Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imaging and\nMultimed

In [46]:
# Split Documents into Smaller Chunks

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20
    )
    text_chunk = text_splitter.split_documents(minimal_docs)
    return text_chunk

In [47]:
text_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(text_chunk)}")

Number of chunks: 5859


In [48]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """

    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )
    return embeddings

embedding = download_embeddings()

In [49]:
embedding

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [50]:
vector = embedding.embed_query("Hello, world")
vector

[-0.05609434098005295,
 0.03554258123040199,
 0.004592902958393097,
 0.02386586368083954,
 -0.04944370687007904,
 -0.15514132380485535,
 0.06592563539743423,
 0.02249131165444851,
 -0.021727291867136955,
 0.014119257219135761,
 0.055051468312740326,
 0.02405569516122341,
 0.005019061733037233,
 -0.00647749612107873,
 -0.03411558270454407,
 -0.05552094429731369,
 -0.006752713583409786,
 -0.023014022037386894,
 -0.17627859115600586,
 -0.023092109709978104,
 1.4232256035029422e-05,
 0.07931112498044968,
 -0.012627873569726944,
 0.037130095064640045,
 -0.09230007231235504,
 -0.023067886009812355,
 0.060699544847011566,
 0.05133027955889702,
 -0.029477505013346672,
 -0.03724546730518341,
 0.037288591265678406,
 0.05159962549805641,
 0.0963367447257042,
 -0.009374106302857399,
 -0.013310282491147518,
 0.0866255834698677,
 -0.08137482404708862,
 -0.06393184512853622,
 0.005632440559566021,
 0.018668074160814285,
 0.05008462443947792,
 -0.0717380940914154,
 -0.05522461608052254,
 -0.0467387661

In [51]:
print(f"Vector length : {len(vector)}")

Vector length : 384


In [52]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [59]:
HF_TOKEN = os.getenv("HF_TOKEN")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")

#PINECONE_HOST = os.getenv("PINECONE_HOST")

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")

LLM_MODEL = os.getenv("LLM_MODEL")

In [60]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key = pinecone_api_key)

In [61]:
pc

In [62]:
from pinecone import ServerlessSpec

index_name = PINECONE_INDEX_NAME

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 384,  # dimension of embeddings
        metric = "cosine", # cosine similarity
        spec = ServerlessSpec(cloud = "aws" , region = "us-east-1")
    )

In [63]:
from pinecone import ServerlessSpec

index = pc.Index(PINECONE_INDEX_NAME)

In [64]:
index

In [65]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = text_chunk,
    embedding = embedding,
    index_name = PINECONE_INDEX_NAME
)

In [66]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upset the embedding into your Pinecone index.

docsearch = PineconeVectorStore.from_existing_index(
    index_name = PINECONE_INDEX_NAME,
    embedding = embedding
)

In [67]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwargs = {"k":3})

In [ ]:
retrived_docs = retriever.invoke("What is Acne?")
retrived_docs

[Document(id='2268faa8-83f6-4afc-9a2b-050fc303cbe6', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='ae74f2f2-8289-41bd-b8a3-8cb00be90a46', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='feda4c45-79fd-4f48-a8b5-83b113032292', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged wi

In [77]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id = LLM_MODEL,
    huggingfacehub_api_token = HF_TOKEN,
    temperature = 0.2,
    max_new_tokens = 512
)

chat_model = ChatHuggingFace(llm = llm)

In [75]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [76]:
system_prompt = (
    "You are an AI Healthcare Assistant that answers questions using only "
    "the provided medical context.\n\n"

    "Instructions:\n"
    "- Use only the information provided in the retrieved context.\n"
    "- If the answer cannot be found in the retrieved context, respond with "
    "'I don't know based on the provided medical information.'\n"
    "- Do not make up facts or provide unsupported medical advice.\n"
    "- Keep your answers accurate, clear, and concise.\n"
    "- Explain medical concepts in simple language whenever possible.\n\n"

    "Retrieved Context:\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [78]:
question_answer_chain = create_stuff_documents_chain(chat_model, prompt)

rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [79]:
response = rag_chain.invoke({"input": "What is Acne?"})

print(response["answer"])

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.


In [80]:
r1 = rag_chain.invoke({"input" : "what is Acne and their treatment"})

print(r1)

{'input': 'what is Acne and their treatment', 'context': [Document(id='1e2349b5-84db-45b0-963c-6e211138e1d1', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'), Document(id='13004b5d-ff40-42a6-919d-e46d312e04e7', metadata={'source': 'data\\Medical_book.pdf'}, page_content='5.\nBergfeld, Wilma F. “The Evaluation and Management of Acne:\nEconomic Considerations.” Journal of the American\nAcademy of Dermatology 32 (1995): S52-6.\nBillings, Laura. “Getting Clear.”Health Magazine, Apr. 1997,\n48-52.\nChristiano, Donna. “Acne Treatment Meant for Grown- Ups.”\nAmerican Health (Oct. 1994): 23-4.\n“Clearly Better New Treatments Help Adult Acne.”Prevention\nMagazine, Aug. 1997, 50-51.\nLeyden, James J. “Therapy For Acne Vulgaris.”New England'), Document(id='80bc8324-a44b-42a8-b63f-f1a51ed4a335', metadata={'source': 'data\\Medical_book.pdf'}, page_content='depends upon whether the acne is mild,